# 1 — Forward Converter Modeling

> **Goal.** Derive the small-signal model of the forward converter,
> show that it's the buck's model with the transformer turns ratio
> $n$ scaling the effective input bus, and explain why this topology
> has **NO RHP zero** despite being isolated.

**Prerequisites**

- Buck modeling notebook (`projects/converters/buck/`). The forward's
  state-space is literally the buck's with one substitution.
- Flyback modeling notebook (for the comparison: same isolation,
  very different control behavior).

**What you'll be able to do at the end**

1. Identify the forward on a schematic and distinguish it from the
   flyback by the energy-flow pattern (forward: simultaneous during
   ON; flyback: store-then-release).
2. Write the switched model and recognize the buck-like structure.
3. Derive $V_o = n \\cdot V_g \\cdot D$ and explain why there's no
   $(1-D)$ in the denominator (no boost-like accumulation).
4. Show that the state-space and transfer functions match the
   buck's verbatim (modulo the $n \\cdot V_g$ substitution).
5. Argue why this topology has the simplest voltage-mode control
   among the four isolated families (forward, push-pull, half-bridge,
   full-bridge).
6. State the **reset-winding constraint** $D \\le 0.5$ (or stricter
   with a non-1:1 reset) and explain how it limits the design space.


## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from forward_model import (
    ForwardParams,
    forward_state_space,
    control_to_output_tf,
    line_to_output_tf,
    output_impedance_tf,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 1. The forward topology

```
V_g+ ---- S ----+
                |
                )||(    ← coupled transformer with reset winding
                )||(    (primary, secondary, reset)
                )||(
                |
               gnd_pri          (isolated barrier)        gnd_sec
                                                              |
                |                          D1 (forward)       |
                |       reset              |--cathode -- L ---+--- V_o
                |       winding            |  to filter        |
                |       (returns           |                   C   R
                |       mag energy         D2 (freewheel)      |   |
                |       to V_g)            |                   |   |
                +-------+-------+         gnd_sec ------------gnd_sec
                                +---- ...
```

- **Primary**: switch $S$ chops $V_g$ at the primary winding.
- **Secondary winding** (during ON): sees $n \\cdot V_g$, drives the
  forward diode $D_1$ → filter inductor $L$ → cap+load. Energy flows
  THROUGH the transformer simultaneously with the switch ON.
- **Freewheel diode** $D_2$ (during OFF): the filter $L$ maintains
  its current; $D_2$ provides the freewheel path. Just like the buck.
- **Reset winding** (during OFF): demagnetizes the transformer.
  Routes the magnetizing energy back to $V_g$ to prepare for the next
  cycle.

### 1.1 Energy routing — forward vs flyback

| Phase | Forward | Flyback |
|---|---|---|
| ON | primary → secondary → L → cap (energy flows through) | primary stores energy in L_m; secondary disconnected |
| OFF | primary off; L freewheels through D₂ | secondary releases stored energy through diode to cap |

The forward is a **transformer + buck**, conceptually. The transformer
is just a voltage scaler — it has no role in energy storage. The buck
filter L-C does the storage.


## 2. Switched (instantaneous) model

State variables: filter inductor current $i_L$ and output cap voltage
$v_o$ (positive, same as the buck — no polarity inversion here).

### 2.1 ON interval ($S$ closed, $D_1$ on, $D_2$ off)

Secondary winding clamps to $n \\cdot v_g$. $D_1$ is forward-biased
(secondary $n v_g$ exceeds $v_o$ initially). $D_2$ is reverse-biased.

The L-C output filter sees a voltage source of $n \\cdot v_g$ on its
input:

$$
L \\cdot \\frac{di_L}{dt} = n \\cdot v_g - v_o
\\qquad
C \\cdot \\frac{dv_o}{dt} = i_L - \\frac{v_o}{R}
$$

### 2.2 OFF interval ($S$ open, $D_1$ off, $D_2$ on)

Primary disconnected. Filter inductor maintains $i_L$; $D_2$ pulls
the L-side of the filter to ground:

$$
L \\cdot \\frac{di_L}{dt} = -v_o
\\qquad
C \\cdot \\frac{dv_o}{dt} = i_L - \\frac{v_o}{R}
$$

Compare to the buck — these equations are **identical** with $v_g$
replaced by $n \\cdot v_g$ on the ON interval. The buck switched
model had $L \\cdot di_L/dt = v_g - v_o$ for ON; the forward has
$L \\cdot di_L/dt = n v_g - v_o$.


## 3. State-space averaging

Average $q \\to d$:

$$\\boxed{
\\;\\; L \\cdot \\frac{di_L}{dt} = d \\cdot n \\cdot v_g - v_o
\\;\\;}
$$

$$\\boxed{
\\;\\; C \\cdot \\frac{dv_o}{dt} = i_L - \\frac{v_o}{R}
\\;\\;}
$$

### 3.1 Steady-state

$$
0 = D \\cdot n \\cdot V_g - V_o
\\;\\implies\\;
\\boxed{V_o = n \\cdot V_g \\cdot D}
$$

$$
0 = I_L - \\frac{V_o}{R}
\\;\\implies\\;
I_L = \\frac{V_o}{R}
$$

The forward output is a **monotonic linear function of duty** — no
$(1-D)$ in the denominator, no "infinite gain as D → 1" singularity.
This is the structural reason the forward has no RHP zero: there's
no inverse relationship between duty and output to invert.

### 3.2 Reset-winding constraint

The transformer's magnetizing flux grows during ON and must reset to
zero each cycle. If the reset winding has the same number of turns
as the primary (1:1 reset), the OFF reset voltage equals $V_g$, and
the flux returns to zero exactly when $(1-D) \\cdot V_g \\cdot T_s = D
\\cdot V_g \\cdot T_s$, giving the constraint:

$$\\boxed{ D_{\\max} = 0.5 \\;\\;\\text{(for 1:1 reset)} }$$

Production designs use $D_{\\max} \\approx 0.45$ to leave margin for
line/load steps and component tolerances.


In [ ]:
params = ForwardParams()
print(operating_point_report(params))
print()
params.assert_reset_winding_ok()
print("✅  Reset-winding constraint check passed.")


## 4. Small-signal linearization

Perturb $i_L = I_L + \\hat i_L$, $v_o = V_o + \\hat v_o$, $d = D + \\hat d$,
$v_g = V_g + \\hat v_g$. Substitute and drop products:

$$
L \\cdot \\frac{d\\hat i_L}{dt}
   = D \\cdot n \\cdot \\hat v_g + n \\cdot V_g \\cdot \\hat d - \\hat v_o
$$

$$
C \\cdot \\frac{d\\hat v_o}{dt}
   = \\hat i_L - \\frac{\\hat v_o}{R}
$$

**Critical observation**: the cap equation has **no $\\hat d$ term**.
That's exactly what gave the buck its lack of RHP zero — and the
forward inherits the property. A positive duty step instantly
delivers more current into the filter inductor → output rises
monotonically.

Compare with the flyback's cap equation:
$C \\cdot d\\hat v_o/dt = (1-D)/n \\cdot \\hat i_L - I_{L_m}/n \\cdot \\hat d - v_o/R$.
The $-I_{L_m}/n \\cdot \\hat d$ term was the source of the flyback's
RHP zero. The forward's transformer doesn't put a $\\hat d$ term in
the cap equation because energy flows are simultaneous, not "stored
then released".


## 5. State-space matrices

$$
A = \\begin{bmatrix}
0 & -1/L \\\\
1/C & -1/(R C)
\\end{bmatrix},
\\quad
B = \\begin{bmatrix}
n V_g / L & n D / L \\\\
0 & 0
\\end{bmatrix}
$$

$$
C = \\begin{bmatrix} 0 & 1 \\end{bmatrix},
\\quad
D_{\\rm feed} = \\begin{bmatrix} 0 & 0 \\end{bmatrix}
$$

Set $n = 1$ → the buck. The transformer's only contribution is the
$n$ multiplier on the duty-and-line-input column of $B$.


In [ ]:
A, B, C_mat, D_mat = forward_state_space(params)
print("A ="); print(A); print()
print("B = [col 0: d̂   col 1: v̂_g]"); print(B); print()
print("C =", C_mat)
print("D (feedthrough) =", D_mat)
print()
eig = np.linalg.eigvals(A)
print(f"A eigenvalues: {eig}")
print(f"Pole magnitude (= ω_n) = {abs(eig[0]):.1f} rad/s "
      f"(expect {params.omega_n:.1f})")


## 6. Transfer functions

### 6.1 $G_{vd}(s)$ — the headline plant

Closed form, identical to the buck with $n V_g$ in place of $V_g$:

$$
G_{vd}(s) = \\frac{n V_g}{LC} \\cdot
\\frac{1}{s^2 + s/(RC) + 1/(LC)}
$$

A clean **second-order low-pass**. **No zeros.** DC gain $= n V_g$,
natural frequency $\\omega_n = 1/\\sqrt{LC}$, $Q = R \\sqrt{C/L}$.

### 6.2 $G_{vg}(s)$, $Z_{out}(s)$

$$
G_{vg}(s) = \\frac{n D}{LC s^2 + L/R \\cdot s + 1}
$$

$$
Z_{out}(s) = \\frac{s L}{LC s^2 + L/R \\cdot s + 1}
$$

Both use the same denominator. $Z_{out}$ is *exactly* the buck's
output impedance — the transformer is invisible to load perturbations.


In [ ]:
Gvd = control_to_output_tf(params)
Gvg = line_to_output_tf(params)
Zout = output_impedance_tf(params)

print(f"Gvd(0)  = {Gvd.num[0] / Gvd.den[2]:.3f} V/duty  "
      f"(expect n·V_g = {params.n*params.V_g:.3f})")
print(f"Gvg(0)  = {Gvg.num[0] / Gvg.den[2]:.4f} V/V    "
      f"(expect n·D = {params.n*params.D:.4f})")
print()
if len(Gvd.num) > 1:
    print(f"Gvd zeros: {np.roots(Gvd.num)}")
else:
    print(f"Gvd zeros: NONE  (no RHP zero — the forward's defining advantage)")
print(f"Gvd poles: {np.roots(Gvd.den)}")


### 6.3 Bode plots — compare with the flyback

Plot $G_{vd}$, $G_{vg}$, and $Z_{out}$. The forward's $G_{vd}$ has
a clean -40 dB/dec rolloff past $f_n$ with phase dropping to a
final -180° — no RHP-zero kink in phase, no magnitude-and-phase
sign inversion. This is what makes forward control "easy".


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2 * np.pi * f

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for tf, name, style in [
    (Gvd,  r"$G_{vd}$ control → output", "-"),
    (Gvg,  r"$G_{vg}$ line → output",    "--"),
    (Zout, r"$Z_{out}$ load → output",   ":"),
]:
    _, mag, ph = signal.bode(tf, w=w)
    ax_mag.semilogx(f, mag, style, label=name)
    ax_ph.semilogx(f, ph, style, label=name)

for ax in (ax_mag, ax_ph):
    ax.axvline(params.f_n, color="C0", linestyle=":", alpha=0.4,
               label=f"$f_n$ = {params.f_n:.0f} Hz")
    ax.axvline(params.f_sw / 10, color="C3", linestyle=":", alpha=0.4,
               label=f"$f_{{sw}}/10$ = {params.f_sw / 10 / 1e3:.0f} kHz (BW target)")
    ax.legend(loc="best", fontsize=8)

ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title(f"Forward open-loop "
                 f"($V_g$={params.V_g}V → $V_o$={params.V_o}V, n={params.n}, "
                 f"D={params.D:.3f})  — NO RHP zero")
plt.tight_layout()
plt.show()


## 7. Step response — monotonic, like the buck

Apply a small duty step. Unlike the flyback / boost / buck-boost,
the forward output rises **monotonically** to its new steady state.
No dip, no inverse response.


In [ ]:
duty_step = 0.01
t = np.linspace(0, 5e-3, 5000)
_, y_step = signal.step(Gvd, T=t)
v_o_pred = params.V_o + duty_step * y_step

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t * 1e3, v_o_pred, label="$v_o$ (analytical small-signal)")
ax.axhline(params.V_o, color="k", linestyle=":", alpha=0.4,
           label=f"Pre-step $V_o$ = {params.V_o} V")
ax.axhline(params.V_o + duty_step * params.n * params.V_g, color="g",
           linestyle=":", alpha=0.5,
           label=f"Predicted new $V_o$ = "
                 f"{params.V_o + duty_step * params.n * params.V_g:.3f} V")
ax.set_xlabel("Time [ms]")
ax.set_ylabel("$v_o$ [V]")
ax.set_title(f"Forward step response: monotonic — no RHP-zero dip")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Pre-step  V_o = {params.V_o:.4f} V")
print(f"Final     v_o = {v_o_pred[-1]:.4f} V")
print(f"Minimum   v_o = {np.min(v_o_pred):.4f} V "
      f"(no dip — monotonic rise)")


## 8. Self-consistency checks

Same three rigorous checks. The ss2tf round-trip needs a small
relative-tolerance trim because scipy can leave numerical noise
(~1e-12 of the dominant coefficient) in the higher-order numerator
slots when the closed-form has fewer terms.


In [ ]:
def _trim_near_zero(coeffs, rel_tol=1e-9):
    '''Strip leading near-zero coefficients (numerical noise from scipy.ss2tf).

    Like np.trim_zeros but with a relative tolerance — needed because
    ss2tf returns a numerator padded to den_degree+1, and the leading
    coefficients may be on the order of 1e-12 of the dominant term
    when the true closed-form numerator has fewer terms.
    '''
    a = np.asarray(coeffs).flatten()
    if a.size == 0:
        return a
    threshold = rel_tol * np.max(np.abs(a))
    nz = np.argmax(np.abs(a) > threshold)
    return a[nz:]


Gvd_closed = control_to_output_tf(params)

# (1) Poles
ss_poles = sorted(np.linalg.eigvals(A), key=lambda z: z.imag)
tf_poles = sorted(np.roots(Gvd_closed.den), key=lambda z: z.imag)
print("(1) Poles:")
print(f"    SS:  {ss_poles}")
print(f"    TF:  {tf_poles}")
pole_match = np.allclose(ss_poles, tf_poles, rtol=1e-10)
print(f"    → match: {pole_match}")

# (2) DC gains
print()
print("(2) DC gains:")
print(f"    Gvd(0)  = {Gvd_closed.num[0] / Gvd_closed.den[2]:8.4f}  "
      f"(expect n·V_g = {params.n * params.V_g:.4f})")
Gvg_local = line_to_output_tf(params)
print(f"    Gvg(0)  = {Gvg_local.num[0] / Gvg_local.den[2]:8.4f}  "
      f"(expect n·D = {params.n * params.D:.4f})")

# (3) ss2tf round-trip (with relative-tolerance trim)
num_from_ss, den_from_ss = signal.ss2tf(A, B, C_mat, D_mat, input=0)
num_from_ss = _trim_near_zero(num_from_ss)
scale_ss = den_from_ss[0]
scale_cf = Gvd_closed.den[0]
num_ss_norm = num_from_ss / scale_ss
num_cf_norm = np.array(Gvd_closed.num) / scale_cf
den_ss_norm = np.array(den_from_ss) / scale_ss
den_cf_norm = np.array(Gvd_closed.den) / scale_cf
print()
print("(3) ss2tf round-trip (trimmed for numerical noise):")
print(f"    Closed-form num = {num_cf_norm}")
print(f"    From-SS num     = {num_ss_norm}")
print(f"    Closed-form den = {den_cf_norm}")
print(f"    From-SS den     = {den_ss_norm}")
round_trip_ok = (
    np.allclose(num_ss_norm, num_cf_norm, rtol=1e-9)
    and np.allclose(den_ss_norm, den_cf_norm, rtol=1e-9)
)
print(f"    → match: {round_trip_ok}")

assert pole_match and round_trip_ok, "Self-consistency check failed!"
print()
print("✅  All three self-consistency checks pass.")


## 9. Summary

The forward converter is the buck with a transformer in front. The
small-signal model is the buck's verbatim, with $n V_g$ replacing
$V_g$ in the $B$ matrix. Voltage-mode control inherits all the
buck's simplicity:

- **No RHP zero** → no bandwidth ceiling beyond $f_{sw}/10$
- **Monotonic step response** → no need for inverse-response
  workarounds
- **Same compensator recipe** as the buck (K-factor Type-III at
  $f_{sw}/10$)

The price for this simplicity is the **reset-winding constraint**:
$D \\le 0.5$ for 1:1 reset, lower for asymmetric reset windings.
Designers handle this by choosing $n$ such that the operating duty
sits comfortably below the cap (typically $D \\le 0.45$).

Math validated by three checks (poles, DC gains, ss2tf round-trip).

**Cross-validation against Pulsim**: open
[`00_forward_pulsim_validation.ipynb`](00_forward_pulsim_validation.ipynb)
for an executed notebook that builds the forward in Pulsim
(transformer + rectifier diode + freewheel diode + LC filter) and
overlays the steady-state output voltage.

**Next**: open `02_forward_controller.ipynb` for the controller
design + switched closed-loop proof.

**Suggested exercises**

1. Push the duty target above 0.5 and run the simulator (turn off
   the safety clip). Watch the magnetizing current run away.
2. Compare $G_{vd}(s)$ for the forward and flyback on the same
   Bode plot at the same $V_g$, $V_o$, $L$, $C$, $R$, $n$.
3. The two-switch forward uses both reset diodes in series with the
   switches to eliminate the reset winding. Where does the analytical
   model change? (Hint: it doesn't — the reset diodes only affect
   the transformer's stress / voltage rating, not the small-signal
   $G_{vd}$.)
